# 02 · Incremental — doc-status pipeline + cross-document entity merging

This reads the `lightrag_news` graph built by `ingest.py` (two waves, tagged `wave-1`/`wave-2`) and inspects the ingestion bookkeeping and a recurring, merged entity.

> Run `ingest.py` first.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
try:
    from lightrag.utils import GRAPH_FIELD_SEP
except Exception:
    GRAPH_FIELD_SEP = "<SEP>"
rag = build_rag("lightrag_news")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
print("entities in graph:", f"{len(await g.get_all_labels()):,}")

entities in graph: 5,493


## The ingestion pipeline (doc-status + per-wave `track_id`)

In [2]:
print("status counts:", await rag.doc_status.get_all_status_counts())
for tid in ("wave-1", "wave-2"):
    print(f"  docs tagged {tid}: {len(await rag.doc_status.get_docs_by_track_id(tid))}")

status counts: {'pending': 0, 'parsing': 0, 'analyzing': 0, 'processing': 0, 'preprocessed': 0, 'processed': 573, 'failed': 28, 'all': 601}
  docs tagged wave-1: 300
  docs tagged wave-2: 300


## A recurring entity, merged across documents

In [3]:
hub = (await g.get_popular_labels(limit=1))[0]
node = await g.get_node(hub)
sources = [s for s in (node.get("source_id") or "").split(GRAPH_FIELD_SEP) if s]
print(f"entity: {hub}")
print(f"  degree: {await g.node_degree(hub)}  source chunks: {len(sources)}")
print(f"  description: {(node.get('description') or '')[:160]}")

entity: Donald Trump
  degree: 85  source chunks: 33
  description: Donald Trump is the 45th President of the United States, having served from January 20, 2017, to January 20, 2021. His presidency was marked by considerable pol


## Doc-status pagination (newest first)

In [4]:
rows, total = await rag.doc_status.get_docs_paginated(page=1, page_size=5,
                                                     sort_field="updated_at", sort_direction="desc")
print("total documents:", total)
for doc_id, st in rows:
    print(f"  {doc_id}  status={getattr(st,'status','?')}  {getattr(st,'file_path','?')}")

total documents: 601
  dup-75084e9bd5b0fea623819b514d7a6233  status=DocStatus.FAILED  mother-daughter-duo-dancing-2516681965.html
  news-623  status=DocStatus.PROCESSED  3385965
  news-624  status=DocStatus.PROCESSED  3354442
  news-622  status=DocStatus.PROCESSED  3413715
  news-609  status=DocStatus.PROCESSED  3354778
  news-621  status=DocStatus.PROCESSED  3354052
  news-615  status=DocStatus.PROCESSED  3353933
  news-620  status=DocStatus.PROCESSED  3444575
  news-619  status=DocStatus.PROCESSED  3474187
  news-618  status=DocStatus.PROCESSED  3188511


## How it was built

`ingest.py` streams CC-News in waves and `ainsert`s each wave with a `track_id`; LightRAG extracts and merges entities across articles, and the doc-status pipeline tracks every document:

```python
await rag.ainsert(texts, ids=ids, file_paths=urls, track_id="wave-1")
```

This notebook reads the finished graph; run `ingest.py` to (re)build it.

In [5]:
await rag.finalize_storages()